# 🔬 Notebook 1 : Exploration & Nettoyage des Données (EDA & Cleaning)
**Projet :** Prédiction de l'Attrition des Employés — Data Mining (Track A : Classification Supervisée)  
**Objectif :** Audit de la qualité des données (10 000 lignes), statistiques descriptives, analyse du déséquilibre de classe, détection et traitement des valeurs aberrantes (outliers IQR), et visualisations univariées/bivariées.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration esthétique
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

print("Bibliothèques chargées avec succès.")

## 1. Chargement et Inspection Initiale du Jeu de Données

In [ ]:
# Chemin vers le fichier brut
data_path = Path("../data/raw/Employee_Attrition_DataSet.csv")
if not data_path.exists():
    data_path = Path("data/raw/Employee_Attrition_DataSet.csv")

df = pd.read_csv(data_path)
print(f"Dimensions du dataset : {df.shape[0]} observations, {df.shape[1]} colonnes.")
df.head()

In [ ]:
# Structure et types des colonnes
df.info()

## 2. Audit de Qualité : Valeurs Manquantes et Doublons

In [ ]:
# Vérification des valeurs manquantes et des doublons
missing = df.isnull().sum()
duplicates = df.duplicated().sum()

print("--- Valeurs manquantes par colonne ---")
print(missing[missing > 0] if missing.sum() > 0 else "Aucune valeur manquante détectée (0 NaN).")
print(f"\nNombre de doublons stricts : {duplicates}")

## 3. Analyse de la Variable Cible : Attrition

In [ ]:
# Distribution de la variable cible
target_counts = df["Attrition"].value_counts()
target_pct = df["Attrition"].value_counts(normalize=True) * 100

summary_target = pd.DataFrame({
    "Effectif": target_counts,
    "Pourcentage (%)": target_pct.round(2)
})
print(summary_target)

# Visualisation du déséquilibre de classe
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Graphique à barres
sns.barplot(x=target_counts.index, y=target_counts.values, palette=["#10B981", "#EF4444"], ax=ax[0])
ax[0].set_title("Distribution des classes (Effectif)")
ax[0].set_ylabel("Nombre d'employés")
for i, v in enumerate(target_counts.values):
    ax[0].text(i, v + 150, f"{v} ({target_pct.iloc[i]:.1f}%)", ha="center", fontweight="bold")

# Graphique en anneau (Donut chart)
ax[1].pie(target_counts, labels=["Stayed (Non)", "Left (Oui)"], colors=["#10B981", "#EF4444"],
          autopct="%1.1f%%", startangle=90, pctdistance=0.85,
          wedgeprops=dict(width=0.4, edgecolor='white'))
ax[1].set_title("Proportion de l'Attrition RH")

plt.tight_layout()
plt.show()

> **Constat :** Le jeu de données présente un fort déséquilibre de classe (~91% Stayed vs ~9% Left, ratio ~10:1). L'Accuracy brute est donc un indicateur trompeur (le classifieur Zero-R atteignant 90.97% sans aucun pouvoir discriminant). Le score F1 et le ROC-AUC seront nos métriques prioritaires.

## 4. Statistiques Descriptives des Variables Numériques

In [ ]:
numerical_cols = [
    "Age", "JobLevel", "YearsAtCompany", "MonthlyIncome",
    "JobSatisfaction", "WorkLifeBalance", "DistanceFromHome",
    "PerformanceRating", "TrainingHoursLastYear"
]

desc_stats = df[numerical_cols].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
desc_stats["IQR"] = desc_stats["75%"] - desc_stats["25%"]
desc_stats.round(2)

## 5. Détection et Traitement des Valeurs Aberrantes (Méthode IQR)

In [ ]:
# Détection des outliers selon la règle 1.5 x IQR
def detect_outliers_iqr(data, columns):
    report = []
    for col in columns:
        q1 = data[col].quantile(0.25)
        q3 = data[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        n_outliers = ((data[col] < lower) | (data[col] > upper)).sum()
        pct = (n_outliers / len(data)) * 100
        report.append({
            "Variable": col,
            "Q1": round(q1, 2),
            "Q3": round(q3, 2),
            "IQR": round(iqr, 2),
            "Borne Inf": round(lower, 2),
            "Borne Sup": round(upper, 2),
            "Nbr Outliers": n_outliers,
            "% Outliers": round(pct, 2)
        })
    return pd.DataFrame(report)

outliers_df = detect_outliers_iqr(df, numerical_cols)
outliers_df

In [ ]:
# Écrêtage (Winsorization) des valeurs aberrantes
def cap_outliers_iqr(data, columns):
    df_capped = data.copy()
    for col in columns:
        q1 = df_capped[col].quantile(0.25)
        q3 = df_capped[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        df_capped[col] = df_capped[col].clip(lower=lower, upper=upper)
    return df_capped

df_cleaned = cap_outliers_iqr(df, numerical_cols)
# Suppression de l'identifiant non prédictif
if "EmployeeID" in df_cleaned.columns:
    df_cleaned = df_cleaned.drop(columns=["EmployeeID"])

print("Jeu de données nettoyé et écrêté avec succès.")
print(f"Dimensions finales : {df_cleaned.shape}")

## 6. Analyse Univariée & Bivariée des Caractéristiques Clés

In [ ]:
# Distributions comparées selon le statut d'attrition
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Revenu mensuel
sns.boxplot(x="Attrition", y="MonthlyIncome", data=df, palette=["#10B981", "#EF4444"], ax=axes[0, 0])
axes[0, 0].set_title("Revenu Mensuel vs Attrition")

# 2. Distance du domicile
sns.boxplot(x="Attrition", y="DistanceFromHome", data=df, palette=["#10B981", "#EF4444"], ax=axes[0, 1])
axes[0, 1].set_title("Distance du Domicile (km) vs Attrition")

# 3. Ancienneté dans l'entreprise
sns.boxplot(x="Attrition", y="YearsAtCompany", data=df, palette=["#10B981", "#EF4444"], ax=axes[1, 0])
axes[1, 0].set_title("Années dans l'Entreprise vs Attrition")

# 4. Overtime (Heures supplémentaires)
ct = pd.crosstab(df["OverTime"], df["Attrition"], normalize="index") * 100
ct.plot(kind="bar", stacked=True, color=["#10B981", "#EF4444"], ax=axes[1, 1], rot=0)
axes[1, 1].set_title("Taux d'Attrition par Heures Supplémentaires (%)")
axes[1, 1].set_ylabel("%")

plt.tight_layout()
plt.show()

## 7. Matrice de Corrélation de Pearson

In [ ]:
# Matrice de corrélation
corr = df[numerical_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Matrice de Corrélation de Pearson (Variables Numériques)")
plt.show()

## 8. Synthèse de l'Étape 1
1. **Intégrité :** 10 000 enregistrements complets, 0 doublons, 0 valeurs manquantes.
2. **Déséquilibre :** 90.97 % `Stayed` vs 9.03 % `Left`.
3. **Outliers :** Présence d'outliers identifiés et traités par bornage IQR sans perte de données.
4. **Relations préliminaires :** Les variables `MonthlyIncome`, `DistanceFromHome`, et `OverTime` montrent de fortes distinctions comportementales entre les employés qui restent et ceux qui partent.